In [1]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
from sklearn.metrics.pairwise import cosine_similarity
import warnings
import re
import ollama
import numpy as np 


C:\Users\Administrator\AppData\Local\Temp\ipykernel_29580\2333973303.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader


In [2]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
from sklearn.metrics.pairwise import cosine_similarity
import warnings
import re
import ollama
import numpy as np 
import chromadb

In [3]:
loader = PyPDFLoader("data/farming.pdf")
pages = loader.load()
chroma_client = chromadb.Client()

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Subtype': '/Type1', '/FontDescriptor': IndirectObject(77, 0, 2041975376256), '/LastChar': 115, '/Widths': [278, 0, 0, 0, 0, 0, 0, 191, 0, 0, 0, 0, 0, 0, 278, 0, 0, 556, 556, 556, 556, 556, 0, 556, 0, 556, 0, 0, 0, 0, 0, 0, 0, 0, 0, 722, 0, 0, 611, 0, 722, 0, 0, 0, 0, 0, 0, 0, 0, 778, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 556, 0, 0, 556, 556, 0, 0, 0, 222, 0, 0, 222, 833, 0, 556, 556, 0, 333, 500], '/BaseFont': '/GFELPE+Helvetica', '/FirstChar': 32, '/Encoding': '/WinAnsiEncoding', '/Type': '/Font'}, but is not installed. Consider installing fontTools if you encounter encoding problems.


#### text Cleaning

In [4]:
def cleanText(text): 
    cleaned_text = re.sub(r'\s+', ' ', text) # clears the spacess
    cleaned_text = cleaned_text.replace('-', " ")
    cleaned_text = cleaned_text.replace(' -', " ")
    cleaned_text = cleaned_text.replace('•', ";")
    cleaned_text = cleaned_text.replace("   ", " ")
    return cleaned_text

In [5]:
for page in pages: 
    page.page_content = cleanText(page.page_content) 

In [6]:
pages[: 10]

[Document(metadata={'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2009-01-23T08:12:14-05:00', 'author': 'holli.kuykendall', 'moddate': '2009-01-23T08:12:45-05:00', 'title': 'Rotations_Soil_Fertility_InfoSheet.pub', 'source': 'data/farming.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='Do you have a problem with: ; Soil crusting ; Cloddy soil ; Water stress (too wet or too dry) for crops ; Soil erosion ; Soil compaction ; Low yields A crop rotation can help to manage your soil and fertility, reduce erosion, improve your soil’s health, and increase nutrients available for crops. Benefits of Crop Rotations: ; Improve crop yields ; Improve the workability of the soil ; Reduce soil crusting ; Increase water available for plants ; Reduce erosion and sedimentation ; Recycle plant nutrients in the soil ; Provide better distribution of labor during the crop season by using different crops, planting dates, and har

#### Chunking
- splitting your text information into smaller sizes

In [7]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 200, chunk_overlap= 100) # created an instance of the text splitter

In [8]:
chunks = splitter.split_documents(pages)

In [9]:
chunks[0:10]

[Document(metadata={'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2009-01-23T08:12:14-05:00', 'author': 'holli.kuykendall', 'moddate': '2009-01-23T08:12:45-05:00', 'title': 'Rotations_Soil_Fertility_InfoSheet.pub', 'source': 'data/farming.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='Do you have a problem with: ; Soil crusting ; Cloddy soil ; Water stress (too wet or too dry) for crops ; Soil erosion ; Soil compaction ; Low yields A crop rotation can help to manage your soil and'),
 Document(metadata={'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2009-01-23T08:12:14-05:00', 'author': 'holli.kuykendall', 'moddate': '2009-01-23T08:12:45-05:00', 'title': 'Rotations_Soil_Fertility_InfoSheet.pub', 'source': 'data/farming.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='; Soil erosion ; Soil compaction ; Low yields A crop rotatio

In [10]:
len(chunks)

49

### Create Embeddings:
- Create embeddings from our chunks
- perform a search:
    - get user input
    - calculate cosine similarity (check for similarity in text/ information)

In [11]:
# functions & global variables 

def createEmbedding(text):
    response = ollama.embed(
    model='nomic-embed-text-v2-moe',
    input=text,
    )
    return response.embeddings

In [12]:
sample_text = "this is a blue car!"
print(createEmbedding(sample_text))

[[-0.0109004555, -0.009896409, -0.10206352, 0.025137765, -0.02900756, -0.01891738, -0.026515814, -0.039523527, -0.021490108, 0.027087031, 0.021864947, -0.08982466, 0.037548967, 0.03867024, 0.02947836, -0.016754854, 0.022226939, -0.014429208, -0.013782003, 0.007815439, 0.06527444, -0.009701454, 0.017374255, -0.06431585, 0.05696782, -0.0033235261, 0.031177137, 0.005097447, 0.042572126, -0.046996072, 0.0155368475, -0.03956419, -0.057435855, 0.06291192, -0.071447216, 0.05664623, -0.021563651, 0.010064815, 0.0049549425, 0.028571278, 0.0060768356, 0.002570171, 0.046614397, 0.058655173, 0.020783614, -0.019239787, 0.009796898, -0.011015159, -0.015904585, -0.0150514, 0.052918702, -0.07825982, 0.022178654, -0.086954646, 0.041563362, -0.01727299, 0.026207918, -0.043715566, 0.023732297, -0.037548196, 0.012715674, -0.0005017516, -0.00068343285, -0.03736228, -0.04365044, -0.04756893, 0.03869052, -0.0126477, 0.02955391, 0.009287542, -0.039796226, 0.035337783, -0.02484911, 0.046461236, -0.018679725, 0

In [13]:
doc_embeddings = [] # this is our list of embeddings from out document

for chunk in chunks:
    chunk_embedding = createEmbedding(chunk.page_content)
    doc_embeddings.append(chunk_embedding) 

In [14]:
len(doc_embeddings)

49

- user asks a question
- embedd the question 
- check for similarity in the document  

In [ ]:
user_query = input("ask me anything about this doc...")

In [ ]:
print(user_query)

benefits of crop rotation


In [ ]:
user_query_embeddding = createEmbedding(user_query)

In [ ]:
print(user_query_embeddding)

[[0.061773922, -0.005446563, 0.013693693, -0.057342526, 0.036096804, -0.0060807727, 0.063145444, 0.02908785, -0.033486076, 0.059857283, -0.040781893, -0.0047990293, -0.042995214, -0.0023736947, -0.02519185, 0.060566857, 0.02812011, -0.038431928, -0.037441976, -0.013549371, -0.0047591343, 0.012933773, -0.03509837, 0.006602795, -0.0044605304, -0.027814988, -0.058526985, -0.0057491995, 0.0328743, -0.03723752, -0.0027658013, -0.046213057, 0.025829302, 0.036290143, 0.0027282462, 0.022156551, -0.11402493, -0.033316553, 0.046397783, 0.008533214, 0.008074379, -0.037633367, 0.03912438, 0.032346528, -0.03651869, 0.015882315, 0.0022721488, -0.01507452, -0.011241963, 0.020071924, -0.018885447, -0.039788622, -0.018563753, -0.033834364, -0.042634662, 0.028417585, 0.013031678, -0.005853952, 0.0603996, -0.028430285, -0.006507196, -0.012161265, -0.03699424, 0.039625064, 0.014569332, -0.026935183, 0.067006424, 0.04452649, 0.003023694, 0.025592865, 0.01175108, 0.03737963, 0.03211063, 0.0491572, -0.090560

In [ ]:
# - loop through the doc_embeedding 
# - calculate the cosine similarity score 
# - return the items with highest score 

above_zero_point_six = []

for index, doc_embedding in enumerate(doc_embeddings):
    score = cosine_similarity(doc_embedding, user_query_embeddding)
    if score >= 0.3:
        result = (index, score)
        above_zero_point_six.append(result)
        # print(score, index)
    

In [ ]:
above_zero_point_six

[(0, array([[0.60649326]])),
 (1, array([[0.73795714]])),
 (2, array([[0.80302147]])),
 (3, array([[0.82356791]])),
 (4, array([[0.44894739]])),
 (5, array([[0.41039128]])),
 (6, array([[0.6575457]])),
 (7, array([[0.64586963]])),
 (8, array([[0.60723689]])),
 (9, array([[0.31670908]])),
 (10, array([[0.54460555]])),
 (12, array([[0.34691711]])),
 (13, array([[0.35108942]])),
 (14, array([[0.35212407]])),
 (15, array([[0.35166254]])),
 (16, array([[0.37200528]])),
 (17, array([[0.42679661]])),
 (18, array([[0.33967802]])),
 (19, array([[0.60854818]])),
 (20, array([[0.5574372]])),
 (21, array([[0.60539199]])),
 (22, array([[0.57299824]])),
 (23, array([[0.35297116]])),
 (24, array([[0.39674859]])),
 (25, array([[0.50466206]])),
 (27, array([[0.5041766]])),
 (28, array([[0.50877023]])),
 (29, array([[0.71330886]])),
 (30, array([[0.6772231]])),
 (31, array([[0.58058565]])),
 (32, array([[0.56619151]])),
 (34, array([[0.33768054]])),
 (35, array([[0.35144448]]))]

In [ ]:
largest =  max(above_zero_point_six, key=lambda x: x[1])
largest

(3, array([[0.82356791]]))

In [ ]:
response_to_client = chunks[largest[0]]
print(response_to_client.page_content)

Benefits of Crop Rotations: ; Improve crop yields ; Improve the workability of the soil ; Reduce soil crusting ; Increase water available for plants ; Reduce erosion and sedimentation ; Recycle plant


In [ ]:
# (RAG) RAG is a brand  new mixed system that claims to be able to fix the issues with pure generative models. 
# There is a system that gets relevant papers or information from an outside source of

In [ ]:
chunks[above_zero_point_six[2][0]].page_content

'fertility, reduce erosion, improve your soil’s health, and increase nutrients available for crops. Benefits of Crop Rotations: ; Improve crop yields ; Improve the workability of the soil ; Reduce'

#### intro to evaluations

In [ ]:
a = np.array([1, 2, 3])
b = np.array([1, 2, 3])

In [ ]:
# Dot Product Similarity
dot_sim = np.dot(a, b)

In [ ]:
# Cosine Similarity
cos_sim = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [ ]:
# Euclidean Distance (Lower is better)
euclidean_dist = np.linalg.norm(a - b)   

In [ ]:
output_dot = f" DOT PRODUCT SIM_SCORE: {dot_sim} "
output_cos = f" COSINE SIM_SCORE: {cos_sim} "
output_eucl = f" EUCLIDEAN DISTANCE: {euclidean_dist} "

print(f""" 
{output_dot} \n 
{output_cos} \n 
{output_eucl} \n 
""")

 
 DOT PRODUCT SIM_SCORE: 14  
 
 COSINE SIM_SCORE: 1.0  
 
 EUCLIDEAN DISTANCE: 0.0  
 



In [ ]:
def similarityChecks(text_one_embeding, text_two_embeding):
    # Dot Product Similarity
    dot_sim = np.dot(text_one_embeding, text_two_embeding)
    # Cosine Similarity
    cos_sim = np.dot(text_one_embeding, text_two_embeding) / (np.linalg.norm(text_one_embeding) * np.linalg.norm(text_two_embeding))
    # Euclidean Distance (Lower is better)
    euclidean_dist = np.linalg.norm(np.array(text_one_embeding) - np.array(text_two_embeding))   

    output_dot = f" DOT PRODUCT SIM_SCORE: {dot_sim} "
    output_cos = f" COSINE SIM_SCORE: {cos_sim} "
    output_eucl = f" EUCLIDEAN DISTANCE: {euclidean_dist} "

    print(f" {output_dot} \n {output_cos} \n {output_eucl} \n ")

In [ ]:
text_one  = "I am driving a blue car"
text_two = "A blue car is being driven by me"

In [ ]:
text_one_embeding = createEmbedding(text=text_one)
text_two_embeding = createEmbedding(text=text_two)

In [ ]:
np.array(text_one_embeding[0][0:10])

array([ 0.01455792, -0.0139812 , -0.08144961, -0.00092482, -0.02538994,
       -0.03063966, -0.02463491, -0.03938487, -0.00139509,  0.0101542 ])

In [ ]:
print(text_one_embeding)
print(text_two_embeding)

[[0.014557918, -0.013981196, -0.081449606, -0.0009248153, -0.025389938, -0.030639656, -0.024634914, -0.039384868, -0.0013950891, 0.010154196, 0.025915336, -0.081142195, -0.0059265173, 0.035093665, 0.031422313, 0.030813755, 0.01823106, 0.03014354, 0.012382759, -0.018471112, 0.025488988, 0.038547926, 0.00057397183, -0.028115116, 0.07139039, -0.011789485, 0.032748297, 0.015907325, 0.03632926, -0.040933654, 0.04211532, -0.014560291, -0.072398104, 0.07264557, -0.0075238612, 0.045316815, -0.03327973, -0.055124898, 0.009934046, 0.014962451, 0.0011419852, 0.025773346, 0.059193287, 0.00080225564, 0.032153726, -0.016031051, 0.01877644, 0.029738357, -0.0020366868, -0.03113033, 0.018945295, -0.08350322, -0.0011875832, -0.039620806, 0.03302715, -0.010333018, 0.005548194, -0.06573134, 0.061073706, -0.028520063, 0.003950013, 0.025559636, -0.01016838, -0.029128686, -0.044582937, -0.048252117, -0.0024905931, -0.009117223, -0.034856677, 0.03853163, -0.0033841939, -0.001177449, -0.04428975, 0.02568356, -

In [ ]:
similarityChecks(text_one_embeding= text_one_embeding[0], text_two_embeding=text_two_embeding[0])

  DOT PRODUCT SIM_SCORE: 0.8640598888118077  
  COSINE SIM_SCORE: 0.8640602796541248  
  EUCLIDEAN DISTANCE: 0.5214204807178836  
 


In [ ]:
user_query = input("ask me anything...")

In [ ]:
"""
    - Ask the user:
        - would you like to add a document? 
        - enter your query based on the document
        - close program
    - Add document then append it to my embeddings list 
    - Take users query -> embeddings -> perform a search in the embedding list 
    - return a score or top k possible responses

    """

'\n    - Ask the user:\n        - would you like to add a document? \n        - enter your query based on the document\n        - close program\n    - Add document then append it to my embeddings list \n    - Take users query -> embeddings -> perform a search in the embedding list \n    - return a score or top k possible responses\n\n    '

## Create your chroma client & Select Model 
- we use persistence to store it in memory such that even when we restart the machine or device we still have the document in place

- **Context** is the text you paste into the prompt so the model can answer from your document.

In [ ]:
chroma_client = chromadb.PersistentClient(path="CHROMA")
MODEL = "llama3.2:3b"
CONTEXT  =""
QUESTION = ""
PERSONA = f""" 
Answer the question using only the context below.
If the context does not contain the answer, say "I don't know based on the document."

Context:
{CONTEXT}

Question: {QUESTION}"""


load your docs

In [ ]:
loader = PyPDFLoader("data/10201979.pdf")
pages = loader.load()

cleaning text

In [ ]:
def cleanText(text):
    text = re.sub(r'-\n(\w)', r'\1', text)     # re-join words hyphenated across lines
    text = text.replace('•', '\n- ')           # bullets as their own lines
    text = re.sub(r'[ \t]+', ' ', text)        # collapse spaces/tabs only
    text = re.sub(r'\n{3,}', '\n\n', text)     # cap blank lines
    return text.strip()

In [ ]:
for page in pages: 
    page.page_content = cleanText(page.page_content)

# Chunking

higher chunk size less granularity 
higher chunk overlap more repetition hence  try and reduce the chunk over lap size to get better context


In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 800, chunk_overlap= 100) # created an instance of the text splitter
chunks = splitter.split_documents(pages)

embeddings

In [ ]:
# functions & global variables 

def createEmbedding(text):
    response = ollama.embed(
    model='nomic-embed-text-v2-moe',
    input=text,
    )
    return response.embeddings[0]

In [ ]:
doc_embeddings = [] # this is our list of embeddings from out document

for chunk in chunks:
    chunk_embedding = createEmbedding(chunk.page_content)
    doc_embeddings.append(chunk_embedding) 

In [ ]:
ids = [ f"10201979:{i}" for i in range(len(chunks))]
documents = [chunk.page_content for chunk in chunks ]
metadatas = [{"source": chunk.metadata.get("source", " "),  }for chunk in chunks ]
embeddings = doc_embeddings


confirming the sizes of each element to assertain consistency

In [ ]:
print(len(ids), len(documents), len(metadatas), len(embeddings))  # all four equal
print(len(embeddings))                                         # single number, e.g. 768

In [ ]:
collection_research = chroma_client.get_or_create_collection(name="CHROMA")

In [ ]:
len(embeddings)

Find out the difference between *upsert* and *add* ad highlight the respective usecase

In [ ]:
collection_research.add(
    ids = ids, 
    documents = documents, 
    metadatas = metadatas, 
    embeddings =  embeddings 
)

In [ ]:
response = collection_research.query(
    query_texts=["What is a recipe?"],
    n_results=2,
    query_embeddings = [createEmbedding("What is a recipe?")],
    # include=["documents", "metadatas", "distances"],
)
 

In [ ]:
response

In [ ]:
response["documents"]

# Retreival 

Recall what we did on pure ollama chat? i.e ollama.chat(model.., messages..., options)? now basd on that we use and guide the model to use our vector DB as out source of truth

In [ ]:
#  query vector DB 
def retreiveFromVectorDB(query, k):
    response = collection_research.query(
    query_texts=[query],
    n_results=k,
    query_embeddings = [createEmbedding(query)],
    # include=["documents", "metadatas", "distances"],
    )
    return response["documents"][0]

def answer(question, k=4):
    chunks = retreiveFromVectorDB(question, k=k)
    CONTEXT = "\n\n---\n\n".join(chunks)  
    QUESTION = question
    prompt = f"""Answer the question using only the context below.
If the context does not contain the answer, say "I don't know based on the document."

Context:
{CONTEXT}

Question: {QUESTION}"""

    resp = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0},   # keeps answers grounded and repeatable (lower chance of hallucination and creativity)
    )
    return resp.message.content


In [ ]:
print(answer("What is a recipe?"))